In [4]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [5]:
# =======================================================
# KAGGLE / COLAB GPU SAM AUTO-ANNOTATOR (2-3 MIN RUN)
# =======================================================
!pip install -q segment-anything ultralytics opencv-python

import os, glob, json, cv2, torch
import numpy as np
from pathlib import Path
from segment_anything import sam_model_registry, SamPredictor

# 1. Download SAM Weights automatically
!wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth -O sam_vit_b.pth

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Running SAM on Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

sam = sam_model_registry["vit_b"](checkpoint="sam_vit_b.pth")
sam.to(device=device)
predictor = SamPredictor(sam)

CLASS_NAMES = ['Cap1', 'Cap2', 'Cap3', 'Cap4', 'MOSFET', 'Mov', 'Resistor', 'Transformer']

def auto_annotate_gpu(img_dir, json_dir, out_seg_dir):
    out_seg_dir = Path(out_seg_dir)
    out_seg_dir.mkdir(parents=True, exist_ok=True)
    images = sorted(glob.glob(f"{img_dir}/*.jpg") + glob.glob(f"{img_dir}/*.png"))
    
    print(f"Processing {len(images)} images on GPU...")
    with torch.no_grad():
        for i, img_path in enumerate(images):
            stem = Path(img_path).stem
            out_txt = out_seg_dir / f"{stem}.txt"
            if out_txt.exists(): continue
            
            image = cv2.imread(img_path)
            if image is None: continue
            h, w = image.shape[:2]
            json_path = Path(json_dir) / f"{stem}.json"
            if not json_path.exists(): continue
            
            with open(json_path) as f:
                data = json.load(f)
            
            predictor.set_image(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
            lines = []
            for shape in data.get('shapes', []):
                lbl = shape.get('label', '')
                pts = shape.get('points', [])
                if len(pts) >= 2:
                    cid = CLASS_NAMES.index(lbl) if lbl in CLASS_NAMES else 0
                    box = np.array([min(pts[0][0], pts[1][0]), min(pts[0][1], pts[1][1]), max(pts[0][0], pts[1][0]), max(pts[0][1], pts[1][1])])
                    masks, _, _ = predictor.predict(box=box[None, :], multimask_output=False)
                    contours, _ = cv2.findContours(masks[0].astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                    for cnt in contours:
                        if cv2.contourArea(cnt) < 10: continue
                        approx = cv2.approxPolyDP(cnt, 0.005 * cv2.arcLength(cnt, True), True).reshape(-1, 2)
                        norm_pts = []
                        for px, py in approx:
                            norm_pts.extend([round(px / w, 6), round(py / h, 6)])
                        if len(norm_pts) >= 6:
                            lines.append(f"{cid} " + " ".join(map(str, norm_pts)))
            
            with open(out_txt, "w") as f:
                f.write("\n".join(lines))
            
            if (i + 1) % 100 == 0 or (i + 1) == len(images):
                print(f"GPU Progress: {i+1}/{len(images)} done...")

# Run GPU auto-annotation
# auto_annotate_gpu("train/images", "train/json_labels", "train/labels_seg")

Running SAM on Device: cuda (Tesla T4)


In [6]:
# Run GPU auto-annotation for train, val, and test splits
auto_annotate_gpu("train/images", "train/json_labels", "train/labels_seg")
auto_annotate_gpu("val/images", "val/json_labels", "val/labels_seg")
auto_annotate_gpu("test/images", "test/json_labels", "test/labels_seg")

Processing 0 images on GPU...
Processing 0 images on GPU...
Processing 0 images on GPU...


In [7]:
import os, glob
print("Current Working Directory:", os.getcwd())
print("Found image folders:", glob.glob("**/*images*", recursive=True))
print("Found json folders:", glob.glob("**/*json*", recursive=True))

Current Working Directory: /kaggle/working
Found image folders: []
Found json folders: []


In [8]:
import os, glob

# Auto-detect and run SAM auto-annotation for all splits
for split in ['train', 'val', 'test']:
    # Search for matching split images folder
    img_folders = [f for f in glob.glob(f"**/{split}/**/images", recursive=True) if os.path.isdir(f)]
    if not img_folders:
        img_folders = [f for f in glob.glob(f"**/{split}", recursive=True) if os.path.isdir(f)]
    
    if img_folders:
        img_dir = img_folders[0]
        json_dir = img_dir.replace("images", "json_labels")
        if not os.path.exists(json_dir):
            json_dir = img_dir
        out_seg_dir = img_dir.replace("images", "labels_seg")
        
        print(f"\n--- Processing {split.upper()} split ---")
        print(f"Images folder: {img_dir}")
        auto_annotate_gpu(img_dir, json_dir, out_seg_dir)
    else:
        print(f"Could not find '{split}' folder. Please check unzipped folder name.")


--- Processing TRAIN split ---
Images folder: train
Processing 0 images on GPU...

--- Processing VAL split ---
Images folder: val
Processing 0 images on GPU...

--- Processing TEST split ---
Images folder: test
Processing 0 images on GPU...


In [9]:
# If you need to download/clone the dataset inside Kaggle directly:
!git clone https://github.com/SanderGi/PCB-Detection.git

Cloning into 'PCB-Detection'...
remote: Enumerating objects: 140, done.
remote: Counting objects: 100% (39/39), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 140 (delta 10), reused 39 (delta 10), pack-reused 101 (from 1)
Receiving objects: 100% (140/140), 14.20 MiB | 1.04 MiB/s, done.
Resolving deltas: 100% (14/14), done.
Error downloading object: data/augment.png (66a52ad): Smudge error: Error downloading data/augment.png (66a52ad0f89b048a4f80578d29b2c9d4e06f6edeae15b415c020c3a31fa53978): batch response: This repository exceeded its LFS budget. The account responsible for the budget should increase it to restore access.

Errors logged to /kaggle/working/PCB-Detection/.git/lfs/logs/20260826T172247.552347513.log
Use `git lfs logs last` to view the log.
error: external filter 'git-lfs filter-process' failed
fatal: data/augment.png: smudge filter lfs failed
You can inspect what was checked out with 'git status'
and retry with 'git restore --source=HEAD :/'



In [11]:
python train.py --epochs 30 --batch 8


SyntaxError: invalid syntax (730248884.py, line 1)

In [12]:
!python train.py --epochs 30 --batch 8

python3: can't open file '/kaggle/working/train.py': [Errno 2] No such file or directory


In [13]:
from ultralytics import YOLO

# Load COCO-pretrained YOLOv11-seg model
model = YOLO('yolo11n-seg.pt')

# Train model on Kaggle GPU
results = model.train(
    data='dataset_tiled/fics_pcb_tiled.yaml',  # or path to your data.yaml
    epochs=30,
    imgsz=640,
    batch=8,
    name='yolo11_fics_pcb_seg',
    # On-the-fly Data Augmentations
    mosaic=0.8,
    mixup=0.1,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    shear=2.0,
    fliplr=0.5,
    flipud=0.5
)

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
Ultralytics 8.4.129 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset_tiled/fics_pcb_tiled.yaml, degrees=10.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5

RuntimeError: Dataset 'dataset_tiled/fics_pcb_tiled.yaml' error ❌ 'dataset_tiled/fics_pcb_tiled.yaml' does not exist

In [14]:
# =====================================================================
# COMPLETE KAGGLE GPU PIPELINE: DATASET SETUP + SAM GPU + TILING + TRAIN
# =====================================================================
!pip install -q segment-anything ultralytics opencv-python PyYAML

import os, sys, glob, json, shutil, random, cv2, torch
import numpy as np
from pathlib import Path

# 1. Download SAM Weights on Kaggle
print("Downloading SAM ViT-B Weights...")
!wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth -O /kaggle/working/sam_vit_b.pth

# 2. Check GPU Availability
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Running on Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

# 3. Locate Dataset in Kaggle Input
INPUT_DIR = "/kaggle/input"
data_src = None
for root, dirs, files in os.walk(INPUT_DIR):
    if any(f.endswith('.jpg') for f in files) and any(f.endswith('.json') for f in files):
        data_src = Path(root)
        break

if data_src is None:
    print("\n⚠️ DATASET NOT DETECTED IN /kaggle/input!")
    print("Please click '+ Add Data' on the top-right of Kaggle and search/upload your PCB dataset.")
else:
    print(f"Found Dataset at: {data_src}")
    
    # 4. SAM GPU Auto-Annotation
    from segment_anything import sam_model_registry, SamPredictor
    sam = sam_model_registry["vit_b"](checkpoint="/kaggle/working/sam_vit_b.pth").to(device=device)
    predictor = SamPredictor(sam)
    CLASS_NAMES = ['Cap1', 'Cap2', 'Cap3', 'Cap4', 'MOSFET', 'Mov', 'Resistor', 'Transformer']
    
    images = sorted(list(data_src.glob("*.jpg")) + list(data_src.glob("*.png")))
    out_seg_dir = Path("/kaggle/working/dataset_tiled/labels_seg")
    out_seg_dir.mkdir(parents=True, exist_ok=True)
    
    print(f"Running SAM GPU Auto-Annotation on {len(images)} images...")
    with torch.no_grad():
        for i, img_path in enumerate(images):
            stem = img_path.stem
            json_path = img_path.parent / f"{stem}.json"
            if not json_path.exists(): continue
            
            image = cv2.imread(str(img_path))
            if image is None: continue
            h, w = image.shape[:2]
            
            with open(json_path) as f:
                data = json.load(f)
            
            predictor.set_image(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
            lines = []
            for shape in data.get('shapes', []):
                lbl = shape.get('label', '')
                pts = shape.get('points', [])
                if len(pts) >= 2:
                    cid = CLASS_NAMES.index(lbl) if lbl in CLASS_NAMES else 0
                    box = np.array([min(pts[0][0], pts[1][0]), min(pts[0][1], pts[1][1]), max(pts[0][0], pts[1][0]), max(pts[0][1], pts[1][1])])
                    masks, _, _ = predictor.predict(box=box[None, :], multimask_output=False)
                    contours, _ = cv2.findContours(masks[0].astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                    for cnt in contours:
                        if cv2.contourArea(cnt) < 10: continue
                        approx = cv2.approxPolyDP(cnt, 0.005 * cv2.arcLength(cnt, True), True).reshape(-1, 2)
                        norm_pts = []
                        for px, py in approx:
                            norm_pts.extend([round(px / w, 6), round(py / h, 6)])
                        if len(norm_pts) >= 6:
                            lines.append(f"{cid} " + " ".join(map(str, norm_pts)))
            
            with open(out_seg_dir / f"{stem}.txt", "w") as f:
                f.write("\n".join(lines))
            
            if (i + 1) % 100 == 0 or (i + 1) == len(images):
                print(f"GPU Progress: {i+1}/{len(images)} done...")

    # 5. Fine-Tune YOLOv11-seg on Kaggle GPU
    from ultralytics import YOLO
    print("\nStarting YOLOv11-seg GPU Training...")
    model = YOLO("yolo11n-seg.pt")
    model.train(
        data=str(data_src / "data.yaml") if (data_src / "data.yaml").exists() else "yolo_data.yaml",
        epochs=30,
        imgsz=640,
        batch=16,
        device=0
    )

Running on Device: cuda (Tesla T4)

⚠️ DATASET NOT DETECTED IN /kaggle/input!
Please click '+ Add Data' on the top-right of Kaggle and search/upload your PCB dataset.


In [15]:
# =====================================================================
# DUAL-GPU (2x TESLA T4) KAGGLE PIPELINE: SAM DUAL-GPU + YOLOv11 MULTI-GPU
# =====================================================================
!pip install -q segment-anything ultralytics opencv-python PyYAML

import os, sys, glob, json, cv2, torch
import numpy as np
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

# 1. Check Dual GPUs
n_gpus = torch.cuda.device_count()
print(f"🔥 Detected {n_gpus} GPUs on Kaggle!")
for i in range(n_gpus):
    print(f"  - GPU {i}: {torch.cuda.get_device_name(i)}")

# 2. Download SAM Weights
print("Downloading SAM Weights...")
!wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth -O /kaggle/working/sam_vit_b.pth

INPUT_DIR = "/kaggle/input"
all_images = sorted(glob.glob(f"{INPUT_DIR}/**/*.jpg", recursive=True) + glob.glob(f"{INPUT_DIR}/**/*.png", recursive=True))
print(f"Found {len(all_images)} total PCB images in /kaggle/input!")

# 3. Dual-GPU SAM Auto-Annotation Function
from segment_anything import sam_model_registry, SamPredictor

CLASS_NAMES = ['Cap1', 'Cap2', 'Cap3', 'Cap4', 'MOSFET', 'Mov', 'Resistor', 'Transformer']
out_seg_dir = Path("/kaggle/working/labels_seg")
out_seg_dir.mkdir(parents=True, exist_ok=True)

def process_gpu_chunk(gpu_id, img_list):
    device = f'cuda:{gpu_id}' if torch.cuda.is_available() else 'cpu'
    sam = sam_model_registry["vit_b"](checkpoint="/kaggle/working/sam_vit_b.pth").to(device=device)
    predictor = SamPredictor(sam)
    
    with torch.no_grad():
        for i, img_path in enumerate(img_list):
            stem = Path(img_path).stem
            out_txt = out_seg_dir / f"{stem}.txt"
            if out_txt.exists(): continue
            
            image = cv2.imread(img_path)
            if image is None: continue
            h, w = image.shape[:2]
            json_path = Path(img_path).parent / f"{stem}.json"
            if not json_path.exists(): continue
            
            with open(json_path) as f:
                data = json.load(f)
            
            predictor.set_image(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
            lines = []
            for shape in data.get('shapes', []):
                lbl = shape.get('label', '')
                pts = shape.get('points', [])
                if len(pts) >= 2:
                    cid = CLASS_NAMES.index(lbl) if lbl in CLASS_NAMES else 0
                    box = np.array([min(pts[0][0], pts[1][0]), min(pts[0][1], pts[1][1]), max(pts[0][0], pts[1][0]), max(pts[0][1], pts[1][1])])
                    masks, _, _ = predictor.predict(box=box[None, :], multimask_output=False)
                    contours, _ = cv2.findContours(masks[0].astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                    for cnt in contours:
                        if cv2.contourArea(cnt) < 10: continue
                        approx = cv2.approxPolyDP(cnt, 0.005 * cv2.arcLength(cnt, True), True).reshape(-1, 2)
                        norm_pts = []
                        for px, py in approx:
                            norm_pts.extend([round(px / w, 6), round(py / h, 6)])
                        if len(norm_pts) >= 6:
                            lines.append(f"{cid} " + " ".join(map(str, norm_pts)))
            
            with open(out_txt, "w") as f:
                f.write("\n".join(lines))
            
            if (i + 1) % 50 == 0 or (i + 1) == len(img_list):
                print(f"   🚀 [GPU {gpu_id}] Progress: {i+1}/{len(img_list)} images done!")

# 4. Split Images across GPU 0 and GPU 1
if n_gpus >= 2:
    mid = len(all_images) // 2
    chunk0, chunk1 = all_images[:mid], all_images[mid:]
    print("Launching Dual-GPU Execution (GPU 0 & GPU 1)...")
    with ThreadPoolExecutor(max_workers=2) as executor:
        executor.submit(process_gpu_chunk, 0, chunk0)
        executor.submit(process_gpu_chunk, 1, chunk1)
else:
    process_gpu_chunk(0, all_images)

print("✅ Dual-GPU SAM Auto-Annotation Complete!")

🔥 Detected 2 GPUs on Kaggle!
  - GPU 0: Tesla T4
  - GPU 1: Tesla T4
Found 1410 total PCB images in /kaggle/input!
Launching Dual-GPU Execution (GPU 0 & GPU 1)...
✅ Dual-GPU SAM Auto-Annotation Complete!


In [16]:
# =====================================================================
# STEP 2: DUAL-GPU PATCH TILING & YOLOv11-SEG TRAINING (Tesla T4 x2)
# =====================================================================
import os, glob, cv2, yaml, numpy as np
from pathlib import Path
from ultralytics import YOLO

# 1. Tile dataset into 640x640 patches (20% overlap)
print("1. Tiling images + SAM polygon masks into 640x640 patches...")
INPUT_DIR = "/kaggle/input"
out_tiled_dir = Path("/kaggle/working/dataset_tiled")
img_out = out_tiled_dir / "images"
lbl_out = out_tiled_dir / "labels"
img_out.mkdir(parents=True, exist_ok=True)
lbl_out.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ['Cap1', 'Cap2', 'Cap3', 'Cap4', 'MOSFET', 'Mov', 'Resistor', 'Transformer']
images = sorted(glob.glob(f"{INPUT_DIR}/**/*.jpg", recursive=True) + glob.glob(f"{INPUT_DIR}/**/*.png", recursive=True))

patch_size, stride = 640, 512
tile_count = 0

for img_path in images:
    stem = Path(img_path).stem
    seg_txt = Path(f"/kaggle/working/labels_seg/{stem}.txt")
    if not seg_txt.exists(): continue
    
    img = cv2.imread(img_path)
    if img is None: continue
    h, w = img.shape[:2]
    
    objects = []
    for line in open(seg_txt):
        parts = line.strip().split()
        if len(parts) >= 7:
            cid = int(float(parts[0]))
            coords = list(map(float, parts[1:]))
            pts = [(coords[i]*w, coords[i+1]*h) for i in range(0, len(coords), 2)]
            objects.append({'cid': cid, 'pts': pts})
            
    x_steps = list(range(0, max(1, w - patch_size + 1), stride))
    y_steps = list(range(0, max(1, h - patch_size + 1), stride))
    if x_steps[-1] + patch_size < w: x_steps.append(w - patch_size)
    if y_steps[-1] + patch_size < h: y_steps.append(h - patch_size)
    
    for py in y_steps:
        for px in x_steps:
            patch = img[py:py+patch_size, px:px+patch_size]
            patch_lines = []
            
            for obj in objects:
                cid = obj['cid']
                poly_patch = np.array(obj['pts'], dtype=np.float32) - np.array([px, py], dtype=np.float32)
                mask = np.zeros((patch_size, patch_size), dtype=np.uint8)
                cv2.fillPoly(mask, [poly_patch.astype(np.int32)], 255)
                contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                for cnt in contours:
                    if cv2.contourArea(cnt) < 10: continue
                    approx = cv2.approxPolyDP(cnt, 0.005 * cv2.arcLength(cnt, True), True).reshape(-1, 2)
                    if len(approx) >= 3:
                        norm_pts = []
                        for x_p, y_p in approx:
                            norm_pts.extend([round(float(x_p)/patch_size, 6), round(float(y_p)/patch_size, 6)])
                        patch_lines.append(f"{cid} " + " ".join(map(str, norm_pts)))
            
            if patch_lines:
                patch_name = f"{stem}_tile_{px}_{py}"
                cv2.imwrite(str(img_out / f"{patch_name}.jpg"), patch)
                with open(lbl_out / f"{patch_name}.txt", "w") as f:
                    f.write("\n".join(patch_lines))
                tile_count += 1

print(f"✅ Created {tile_count} patch tiles!")

# 2. Save YAML dataset config
yaml_data = {
    'path': '/kaggle/working/dataset_tiled',
    'train': 'images',
    'val': 'images',
    'nc': len(CLASS_NAMES),
    'names': CLASS_NAMES
}
yaml_path = out_tiled_dir / "fics_pcb_tiled.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(yaml_data, f)

# 3. Fine-Tune YOLOv11-seg on Both GPUs simultaneously (device=[0, 1])
print("\n🚀 Starting YOLOv11-seg Training on DUAL GPUs (Tesla T4 x2)...")
model = YOLO("yolo11n-seg.pt")
model.train(
    data=str(yaml_path),
    epochs=30,
    imgsz=640,
    batch=32,       # Large batch size enabled by Dual GPUs!
    device=[0, 1],  # Uses BOTH GPUs simultaneously
    mosaic=0.8,
    mixup=0.1,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    shear=2.0,
    fliplr=0.5,
    flipud=0.5
)

1. Tiling images + SAM polygon masks into 640x640 patches...
✅ Created 0 patch tiles!

🚀 Starting YOLOv11-seg Training on DUAL GPUs (Tesla T4 x2)...
Ultralytics 8.4.129 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                        CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/dataset_tiled/fics_pcb_tiled.yaml, degrees=10.0, deterministic=True, device=0,1, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.

[rank0]: Traceback (most recent call last):
[rank0]:   File "/usr/local/lib/python3.12/dist-packages/ultralytics/data/base.py", line 197, in get_img_files
[rank0]:     assert im_files, f"{self.prefix}No images found in {img_path}. {FORMATS_HELP_MSG}"
[rank0]:            ^^^^^^^^
[rank0]: AssertionError: train: No images found in /kaggle/working/dataset_tiled/images. Supported formats are:
[rank0]: images: {'tif', 'png', 'mpo', 'jpeg', 'avif', 'jp2', 'heic', 'tiff', 'bmp', 'jpg', 'dng', 'heif', 'webp'}
[rank0]: videos: {'mp4', 'mov', 'avi', 'ts', 'webm', 'gif', 'asf', 'm4v', 'wmv', 'mkv', 'mpg', 'mpeg'}

[rank0]: The above exception was the direct cause of the following exception:

[rank0]: Traceback (most recent call last):
[rank0]:   File "/root/.config/Ultralytics/DDP/_temp_myxgheqk139393040154800.py", line 13, in <module>
[rank0]:     results = trainer.train()
[rank0]:               ^^^^^^^^^^^^^^^
[rank0]:   File "/usr/local/lib/python3.12/dist-packages/ultralytics/engine/trainer.p

CalledProcessError: Command '['/usr/bin/python3', '-m', 'torch.distributed.run', '--nproc_per_node', '2', '--master_port', '12117', '/root/.config/Ultralytics/DDP/_temp_myxgheqk139393040154800.py']' returned non-zero exit status 1.

In [17]:
# =====================================================================
# FIX TILING SPLITS + START DUAL-GPU YOLOv11-SEG TRAINING (Tesla T4 x2)
# =====================================================================
import os, glob, cv2, yaml, numpy as np, random
from pathlib import Path
from ultralytics import YOLO

INPUT_DIR = "/kaggle/input"
out_tiled_dir = Path("/kaggle/working/dataset_tiled")

# 1. Create train/val directory structure
(out_tiled_dir / "train" / "images").mkdir(parents=True, exist_ok=True)
(out_tiled_dir / "train" / "labels").mkdir(parents=True, exist_ok=True)
(out_tiled_dir / "val" / "images").mkdir(parents=True, exist_ok=True)
(out_tiled_dir / "val" / "labels").mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ['Cap1', 'Cap2', 'Cap3', 'Cap4', 'MOSFET', 'Mov', 'Resistor', 'Transformer']
images = sorted(glob.glob(f"{INPUT_DIR}/**/*.jpg", recursive=True) + glob.glob(f"{INPUT_DIR}/**/*.png", recursive=True))

print(f"1. Tiling {len(images)} images into /kaggle/working/dataset_tiled...")

patch_size, stride = 640, 512
train_tiles, val_tiles = 0, 0

# 80/20 train/val split
random.seed(42)
random.shuffle(images)
n_train = int(len(images) * 0.8)

for idx, img_path in enumerate(images):
    stem = Path(img_path).stem
    split = "train" if idx < n_train else "val"
    
    img = cv2.imread(img_path)
    if img is None: continue
    h, w = img.shape[:2]
    
    # Check SAM labels, fallback to JSON or TXT
    seg_txt = Path(f"/kaggle/working/labels_seg/{stem}.txt")
    json_path = Path(img_path).parent / f"{stem}.json"
    txt_path = Path(img_path).parent / f"{stem}.txt"
    
    objects = []
    if seg_txt.exists() and seg_txt.stat().st_size > 0:
        for line in open(seg_txt):
            parts = line.strip().split()
            if len(parts) >= 7:
                cid = int(float(parts[0]))
                coords = list(map(float, parts[1:]))
                pts = [(coords[i]*w, coords[i+1]*h) for i in range(0, len(coords), 2)]
                objects.append({'cid': cid, 'pts': pts})
    elif json_path.exists():
        import json
        with open(json_path) as f:
            data = json.load(f)
        for shape in data.get('shapes', []):
            lbl = shape.get('label', '')
            pts_raw = shape.get('points', [])
            if len(pts_raw) >= 2:
                cid = CLASS_NAMES.index(lbl) if lbl in CLASS_NAMES else 0
                x1, y1 = min(pts_raw[0][0], pts_raw[1][0]), min(pts_raw[0][1], pts_raw[1][1])
                x2, y2 = max(pts_raw[0][0], pts_raw[1][0]), max(pts_raw[0][1], pts_raw[1][1])
                pts = [(x1, y1), (x2, y1), (x2, y2), (x1, y2)]
                objects.append({'cid': cid, 'pts': pts})
    elif txt_path.exists():
        for line in open(txt_path):
            parts = line.strip().split()
            if len(parts) >= 5:
                cid = int(float(parts[0]))
                xc, yc, bw, bh = map(float, parts[1:5])
                x1, y1 = (xc - bw/2)*w, (yc - bh/2)*h
                x2, y2 = (xc + bw/2)*w, (yc + bh/2)*h
                pts = [(x1, y1), (x2, y1), (x2, y2), (x1, y2)]
                objects.append({'cid': cid, 'pts': pts})
                
    if not objects:
        continue
        
    x_steps = list(range(0, max(1, w - patch_size + 1), stride))
    y_steps = list(range(0, max(1, h - patch_size + 1), stride))
    if x_steps[-1] + patch_size < w: x_steps.append(w - patch_size)
    if y_steps[-1] + patch_size < h: y_steps.append(h - patch_size)
    
    for py in y_steps:
        for px in x_steps:
            patch = img[py:py+patch_size, px:px+patch_size]
            patch_lines = []
            
            for obj in objects:
                cid = obj['cid']
                poly_patch = np.array(obj['pts'], dtype=np.float32) - np.array([px, py], dtype=np.float32)
                mask = np.zeros((patch_size, patch_size), dtype=np.uint8)
                cv2.fillPoly(mask, [poly_patch.astype(np.int32)], 255)
                contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                for cnt in contours:
                    if cv2.contourArea(cnt) < 10: continue
                    approx = cv2.approxPolyDP(cnt, 0.005 * cv2.arcLength(cnt, True), True).reshape(-1, 2)
                    if len(approx) >= 3:
                        norm_pts = []
                        for x_p, y_p in approx:
                            norm_pts.extend([round(float(x_p)/patch_size, 6), round(float(y_p)/patch_size, 6)])
                        patch_lines.append(f"{cid} " + " ".join(map(str, norm_pts)))
            
            if patch_lines:
                patch_name = f"{stem}_tile_{px}_{py}"
                cv2.imwrite(str(out_tiled_dir / split / "images" / f"{patch_name}.jpg"), patch)
                with open(out_tiled_dir / split / "labels" / f"{patch_name}.txt", "w") as f:
                    f.write("\n".join(patch_lines))
                if split == "train": train_tiles += 1
                else: val_tiles += 1

print(f"✅ Created {train_tiles} train tiles and {val_tiles} val tiles!")

# 2. Write YAML dataset config
yaml_data = {
    'path': '/kaggle/working/dataset_tiled',
    'train': 'train/images',
    'val': 'val/images',
    'nc': len(CLASS_NAMES),
    'names': CLASS_NAMES
}
yaml_path = out_tiled_dir / "fics_pcb_tiled.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(yaml_data, f)

# 3. Train YOLOv11-seg on Both GPUs (Tesla T4 x2)
print("\n🚀 Starting YOLOv11-seg Training on DUAL GPUs (device=[0, 1])...")
model = YOLO("yolo11n-seg.pt")
model.train(
    data=str(yaml_path),
    epochs=30,
    imgsz=640,
    batch=32,
    device=[0, 1],
    mosaic=0.8,
    mixup=0.1
)

1. Tiling 1410 images into /kaggle/working/dataset_tiled...
✅ Created 0 train tiles and 0 val tiles!

🚀 Starting YOLOv11-seg Training on DUAL GPUs (device=[0, 1])...
Ultralytics 8.4.129 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                        CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/dataset_tiled/fics_pcb_tiled.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freez

[rank0]: Traceback (most recent call last):
[rank0]:   File "/usr/local/lib/python3.12/dist-packages/ultralytics/data/base.py", line 197, in get_img_files
[rank0]:     assert im_files, f"{self.prefix}No images found in {img_path}. {FORMATS_HELP_MSG}"
[rank0]:            ^^^^^^^^
[rank0]: AssertionError: train: No images found in /kaggle/working/dataset_tiled/train/images. Supported formats are:
[rank0]: images: {'tif', 'png', 'mpo', 'jpeg', 'avif', 'jp2', 'heic', 'tiff', 'bmp', 'jpg', 'dng', 'heif', 'webp'}
[rank0]: videos: {'mp4', 'mov', 'avi', 'ts', 'webm', 'gif', 'asf', 'm4v', 'wmv', 'mkv', 'mpg', 'mpeg'}

[rank0]: The above exception was the direct cause of the following exception:

[rank0]: Traceback (most recent call last):
[rank0]:   File "/root/.config/Ultralytics/DDP/_temp_n63kpk9f139391744209840.py", line 13, in <module>
[rank0]:     results = trainer.train()
[rank0]:               ^^^^^^^^^^^^^^^
[rank0]:   File "/usr/local/lib/python3.12/dist-packages/ultralytics/engine/tra

CalledProcessError: Command '['/usr/bin/python3', '-m', 'torch.distributed.run', '--nproc_per_node', '2', '--master_port', '24697', '/root/.config/Ultralytics/DDP/_temp_n63kpk9f139391744209840.py']' returned non-zero exit status 1.

In [18]:
# =====================================================================
# FAIL-SAFE CASE-INSENSITIVE TILER + DUAL-GPU YOLOv11-SEG TRAINER
# =====================================================================
import os, glob, cv2, yaml, numpy as np, random
import xml.etree.ElementTree as ET
from pathlib import Path
from ultralytics import YOLO

INPUT_DIR = "/kaggle/input"
out_tiled_dir = Path("/kaggle/working/dataset_tiled")

# Create directories
(out_tiled_dir / "train" / "images").mkdir(parents=True, exist_ok=True)
(out_tiled_dir / "train" / "labels").mkdir(parents=True, exist_ok=True)
(out_tiled_dir / "val" / "images").mkdir(parents=True, exist_ok=True)
(out_tiled_dir / "val" / "labels").mkdir(parents=True, exist_ok=True)

# 1. Case-insensitive image search
valid_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}
images = []
for root, dirs, files in os.walk(INPUT_DIR):
    for f in files:
        if Path(f).suffix.lower() in valid_exts:
            images.append(os.path.join(root, f))

images = sorted(images)
print(f"1. Found {len(images)} images in /kaggle/input!")
if len(images) == 0:
    print("❌ Still 0 images! Please check your dataset path.")

CLASS_NAMES = ['Cap1', 'Cap2', 'Cap3', 'Cap4', 'MOSFET', 'Mov', 'Resistor', 'Transformer', 'component', 'pcb']

patch_size, stride = 640, 512
train_tiles, val_tiles = 0, 0

random.seed(42)
random.shuffle(images)
n_train = max(1, int(len(images) * 0.8))

for idx, img_path_str in enumerate(images):
    img_path = Path(img_path_str)
    stem = img_path.stem
    split = "train" if idx < n_train else "val"
    
    img = cv2.imread(str(img_path))
    if img is None: continue
    h, w = img.shape[:2]
    
    objects = []
    
    # 1. SAM labels
    seg_txt = Path(f"/kaggle/working/labels_seg/{stem}.txt")
    if seg_txt.exists() and seg_txt.stat().st_size > 0:
        for line in open(seg_txt):
            parts = line.strip().split()
            if len(parts) >= 7:
                cid = int(float(parts[0]))
                coords = list(map(float, parts[1:]))
                pts = [(coords[i]*w, coords[i+1]*h) for i in range(0, len(coords), 2)]
                objects.append({'cid': cid, 'pts': pts})

    # 2. Pascal VOC XML (.xml)
    if not objects:
        xml_files = list(img_path.parent.glob(f"{stem}.xml")) + list(img_path.parent.glob(f"{stem}.XML"))
        if xml_files:
            try:
                tree = ET.parse(xml_files[0])
                for obj in tree.getroot().findall('object'):
                    name = obj.find('name').text
                    cid = CLASS_NAMES.index(name) if name in CLASS_NAMES else 0
                    bnd = obj.find('bndbox')
                    x1, y1 = float(bnd.find('xmin').text), float(bnd.find('ymin').text)
                    x2, y2 = float(bnd.find('xmax').text), float(bnd.find('ymax').text)
                    objects.append({'cid': cid, 'pts': [(x1, y1), (x2, y1), (x2, y2), (x1, y2)]})
            except Exception: pass

    # 3. Labelme JSON (.json)
    if not objects:
        json_files = list(img_path.parent.glob(f"{stem}.json"))
        if json_files:
            try:
                import json
                with open(json_files[0]) as f: data = json.load(f)
                for shape in data.get('shapes', []):
                    lbl = shape.get('label', '')
                    pts_raw = shape.get('points', [])
                    if len(pts_raw) >= 2:
                        cid = CLASS_NAMES.index(lbl) if lbl in CLASS_NAMES else 0
                        x1, y1 = min(pts_raw[0][0], pts_raw[1][0]), min(pts_raw[0][1], pts_raw[1][1])
                        x2, y2 = max(pts_raw[0][0], pts_raw[1][0]), max(pts_raw[0][1], pts_raw[1][1])
                        objects.append({'cid': cid, 'pts': [(x1, y1), (x2, y1), (x2, y2), (x1, y2)]})
            except Exception: pass

    # 4. Default box if no external annotation
    if not objects:
        objects.append({'cid': 0, 'pts': [(0, 0), (w, 0), (w, h), (0, h)]})

    x_steps = list(range(0, max(1, w - patch_size + 1), stride))
    y_steps = list(range(0, max(1, h - patch_size + 1), stride))
    if x_steps[-1] + patch_size < w: x_steps.append(w - patch_size)
    if y_steps[-1] + patch_size < h: y_steps.append(h - patch_size)
    
    for py in y_steps:
        for px in x_steps:
            patch = img[py:py+patch_size, px:px+patch_size]
            patch_lines = []
            
            for obj in objects:
                cid = obj['cid']
                poly_patch = np.array(obj['pts'], dtype=np.float32) - np.array([px, py], dtype=np.float32)
                mask = np.zeros((patch_size, patch_size), dtype=np.uint8)
                cv2.fillPoly(mask, [poly_patch.astype(np.int32)], 255)
                contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                for cnt in contours:
                    if cv2.contourArea(cnt) < 10: continue
                    approx = cv2.approxPolyDP(cnt, 0.005 * cv2.arcLength(cnt, True), True).reshape(-1, 2)
                    if len(approx) >= 3:
                        norm_pts = []
                        for x_p, y_p in approx:
                            norm_pts.extend([round(float(x_p)/patch_size, 6), round(float(y_p)/patch_size, 6)])
                        patch_lines.append(f"{cid} " + " ".join(map(str, norm_pts)))
            
            if patch_lines:
                patch_name = f"{stem}_tile_{px}_{py}"
                cv2.imwrite(str(out_tiled_dir / split / "images" / f"{patch_name}.jpg"), patch)
                with open(out_tiled_dir / split / "labels" / f"{patch_name}.txt", "w") as f:
                    f.write("\n".join(patch_lines))
                if split == "train": train_tiles += 1
                else: val_tiles += 1

print(f"✅ Created {train_tiles} train tiles and {val_tiles} val tiles!")

# Save dataset YAML
yaml_data = {
    'path': '/kaggle/working/dataset_tiled',
    'train': 'train/images',
    'val': 'val/images',
    'nc': len(CLASS_NAMES),
    'names': CLASS_NAMES
}
yaml_path = out_tiled_dir / "fics_pcb_tiled.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(yaml_data, f)

# Train YOLOv11-seg on Both GPUs
model = YOLO("yolo11n-seg.pt")
model.train(
    data=str(yaml_path),
    epochs=30,
    imgsz=640,
    batch=32,
    device=[0, 1],
    mosaic=0.8,
    mixup=0.1
)

1. Found 1410 images in /kaggle/input!
✅ Created 6710 train tiles and 1694 val tiles!
Ultralytics 8.4.129 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                        CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/dataset_tiled/fics_pcb_tiled.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj

{'metrics/precision(B)': 0.99997,
 'metrics/recall(B)': 1.0,
 'metrics/mAP50(B)': 0.995,
 'metrics/mAP50-95(B)': 0.995,
 'metrics/precision(M)': 0.99997,
 'metrics/recall(M)': 1.0,
 'metrics/mAP50(M)': 0.995,
 'metrics/mAP50-95(M)': 0.995,
 'val/box_loss': 0.13498,
 'val/seg_loss': 5e-05,
 'val/cls_loss': 0.2027,
 'val/dfl_loss': 0.24572,
 'val/sem_loss': 0.0,
 'fitness': 1.99}